# 9. Policy Gradient Methods

Bu notebook, Sutton & Barto kitabının 13. bölümünü kapsar.

## İçindekiler
1. Policy Gradient'a Giriş
2. Policy Gradient Theorem
3. REINFORCE Algorithm
4. REINFORCE with Baseline
5. Actor-Critic Methods

## 9.1 Neden Policy Gradient?

### Value-based vs Policy-based

**Value-based** (Q-Learning, SARSA):
- Q(s, a) öğren, policy'yi dolaylı olarak türet
- $\pi(s) = \arg\max_a Q(s, a)$
- Deterministic policy

**Policy-based** (Policy Gradient):
- Policy'yi **doğrudan** parametrize et ve öğren
- $\pi_\theta(a|s)$ - stochastic policy
- Continuous action spaces için doğal

### Policy Gradient'ın Avantajları

1. **Stochastic policies**: Doğal olarak explore eder
2. **Continuous actions**: Kolay parametrizasyon
3. **Smoother optimization**: Küçük θ değişimi → küçük policy değişimi
4. **Teorik garanti**: Local optimum'a yakınsama

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

class ShortCorridorEnv:
    """
    Short Corridor with Switched Actions (Example 13.1).
    
    States: 0, 1, 2, 3 (3 is terminal)
    Actions: left (0), right (1)
    
    Twist: In state 1, actions are reversed!
    """
    
    def __init__(self):
        self.n_states = 4
        self.n_actions = 2
        self.start_state = 0
        self.terminal_state = 3
        self.reset()
    
    def reset(self):
        self.state = self.start_state
        return self.state
    
    def step(self, action):
        # In state 1, actions are reversed
        if self.state == 1:
            action = 1 - action  # Flip action
        
        if action == 0:  # Left
            self.state = max(0, self.state - 1)
        else:  # Right
            self.state = min(3, self.state + 1)
        
        done = (self.state == self.terminal_state)
        reward = -1
        
        return self.state, reward, done

env = ShortCorridorEnv()
print("Short Corridor Environment")
print("States: 0 -> 1 -> 2 -> 3 (terminal)")
print("In state 1, actions are reversed!")

## 9.2 Policy Parametrization

### Softmax Policy

En yaygın parametrizasyon:

$$\pi_\theta(a|s) = \frac{e^{h(s, a, \theta)}}{\sum_b e^{h(s, b, \theta)}}$$

Burada $h(s, a, \theta)$ action preference function.

### Linear Softmax

$$h(s, a, \theta) = \theta^T \phi(s, a)$$

Basit durumda: $h(s, a, \theta) = \theta_a$ (state-independent)

In [ ]:
def softmax(x):
    """Numerically stable softmax."""
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

class SoftmaxPolicy:
    """
    Simple softmax policy with linear preferences.
    For Short Corridor: state-independent, only 2 actions.
    θ = single parameter controlling P(right)
    """
    
    def __init__(self, n_actions=2):
        self.n_actions = n_actions
        # θ: preference for right action (relative to left)
        self.theta = np.array([0.0])  # h(left)=0, h(right)=θ
    
    def get_probs(self, state=None):
        """Return action probabilities."""
        preferences = np.array([0.0, self.theta[0]])
        return softmax(preferences)
    
    def select_action(self, state):
        probs = self.get_probs(state)
        return np.random.choice(self.n_actions, p=probs)
    
    def get_grad_log_prob(self, state, action):
        """
        ∇_θ log π(a|s)
        
        For softmax: ∇_θ log π(a|s) = φ(s,a) - Σ_b π(b|s) φ(s,b)
        
        Here: φ(s, left) = 0, φ(s, right) = 1
        So: ∇_θ log π(a|s) = I(a=right) - π(right|s)
        """
        probs = self.get_probs(state)
        # Feature for action: 0 for left, 1 for right
        feature = 1.0 if action == 1 else 0.0
        grad = feature - probs[1]
        return np.array([grad])

# Test
policy = SoftmaxPolicy()
print(f"θ = {policy.theta[0]:.2f}")
print(f"P(left) = {policy.get_probs()[0]:.3f}")
print(f"P(right) = {policy.get_probs()[1]:.3f}")

## 9.3 Policy Gradient Theorem

### Objective Function

$$J(\theta) = V^{\pi_\theta}(s_0)$$

veya ortalama reward:

$$J(\theta) = \sum_s d^\pi(s) V^\pi(s)$$

### Policy Gradient Theorem

$$\nabla J(\theta) \propto \sum_s d^\pi(s) \sum_a Q^\pi(s, a) \nabla_\theta \pi_\theta(a|s)$$

veya equivalently:

$$\nabla J(\theta) = E_{\pi}[Q^\pi(s, a) \nabla_\theta \log \pi_\theta(a|s)]$$

Bu, **sample-based gradient**'ı mümkün kılar!

## 9.4 REINFORCE: Monte Carlo Policy Gradient

$Q^\pi(s, a)$'yı sample return $G_t$ ile tahmin et:

$$\theta_{t+1} = \theta_t + \alpha G_t \nabla_\theta \log \pi_\theta(A_t|S_t)$$

### Algoritma

```
Initialize θ
For each episode:
    Generate episode S_0, A_0, R_1, ..., S_T following π_θ
    For each step t:
        G_t ← return from step t
        θ ← θ + α γ^t G_t ∇_θ log π_θ(A_t|S_t)
```

In [ ]:
def reinforce(env, policy, n_episodes=1000, alpha=2e-4, gamma=1.0):
    """
    REINFORCE Algorithm (Monte Carlo Policy Gradient).
    """
    episode_rewards = []
    
    for episode in range(n_episodes):
        # Generate episode
        states, actions, rewards = [], [], []
        state = env.reset()
        
        while True:
            action = policy.select_action(state)
            states.append(state)
            actions.append(action)
            
            next_state, reward, done = env.step(action)
            rewards.append(reward)
            
            if done:
                break
            state = next_state
        
        episode_rewards.append(sum(rewards))
        
        # Calculate returns
        T = len(rewards)
        returns = np.zeros(T)
        G = 0
        for t in range(T - 1, -1, -1):
            G = rewards[t] + gamma * G
            returns[t] = G
        
        # Update policy
        for t in range(T):
            grad = policy.get_grad_log_prob(states[t], actions[t])
            policy.theta += alpha * (gamma ** t) * returns[t] * grad
    
    return episode_rewards

# Train REINFORCE
policy = SoftmaxPolicy()
rewards = reinforce(env, policy, n_episodes=1000, alpha=2e-4)

print(f"Final θ = {policy.theta[0]:.4f}")
print(f"P(right) = {policy.get_probs()[1]:.4f}")

In [ ]:
# Plot learning curve
plt.figure(figsize=(12, 5))

# Smooth rewards
window = 50
smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')

plt.subplot(1, 2, 1)
plt.plot(smoothed)
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('REINFORCE Learning Curve')
plt.axhline(y=-11.6, color='r', linestyle='--', label='Optimal (≈-11.6)')
plt.legend()
plt.grid(True, alpha=0.3)

# Show optimal P(right) analysis
plt.subplot(1, 2, 2)
p_rights = np.linspace(0.01, 0.99, 100)

# Expected return for different P(right)
# This requires analysis of the environment
def expected_return(p_right):
    # Simplified: approximate expected steps to goal
    # State 1 has reversed actions, making higher P(right) better
    # True optimal is around p_right = 0.59
    return -1 / (2 * p_right - 1 + 0.01) if p_right > 0.5 else -20

expected_returns = [expected_return(p) for p in p_rights]
plt.plot(p_rights, expected_returns)
plt.axvline(x=policy.get_probs()[1], color='r', linestyle='--', label=f'Learned: {policy.get_probs()[1]:.2f}')
plt.xlabel('P(right)')
plt.ylabel('Expected Return (approx)')
plt.title('Expected Return vs P(right)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9.5 REINFORCE with Baseline

REINFORCE yüksek variance'a sahip. **Baseline** ekleyerek azaltabiliriz:

$$\nabla J(\theta) = E_{\pi}[(G_t - b(S_t)) \nabla_\theta \log \pi_\theta(A_t|S_t)]$$

Baseline $b(s)$ **bias** eklemez (çünkü $E[\nabla \log \pi \cdot b] = 0$), ama variance'ı azaltır.

### En İyi Baseline

$$b(s) = V^\pi(s)$$

Bu durumda $G_t - b(S_t) \approx A^\pi(S_t, A_t)$ (**Advantage**).

In [ ]:
class ValueFunction:
    """
    Simple state value function approximator.
    """
    
    def __init__(self, n_states):
        self.w = np.zeros(n_states)  # One weight per state
    
    def __call__(self, state):
        return self.w[state]
    
    def update(self, state, target, alpha=0.1):
        self.w[state] += alpha * (target - self.w[state])

def reinforce_baseline(env, policy, n_episodes=1000, alpha_policy=2e-4, 
                       alpha_value=0.1, gamma=1.0):
    """
    REINFORCE with Baseline.
    """
    V = ValueFunction(env.n_states)
    episode_rewards = []
    
    for episode in range(n_episodes):
        # Generate episode
        states, actions, rewards = [], [], []
        state = env.reset()
        
        while True:
            action = policy.select_action(state)
            states.append(state)
            actions.append(action)
            
            next_state, reward, done = env.step(action)
            rewards.append(reward)
            
            if done:
                break
            state = next_state
        
        episode_rewards.append(sum(rewards))
        
        # Calculate returns
        T = len(rewards)
        returns = np.zeros(T)
        G = 0
        for t in range(T - 1, -1, -1):
            G = rewards[t] + gamma * G
            returns[t] = G
        
        # Update policy and value function
        for t in range(T):
            s, a = states[t], actions[t]
            
            # Advantage: G_t - V(S_t)
            delta = returns[t] - V(s)
            
            # Update value function
            V.update(s, returns[t], alpha_value)
            
            # Update policy
            grad = policy.get_grad_log_prob(s, a)
            policy.theta += alpha_policy * (gamma ** t) * delta * grad
    
    return episode_rewards

# Compare REINFORCE with and without baseline
n_runs = 20
n_episodes = 1000

results_no_baseline = []
results_baseline = []

for _ in range(n_runs):
    policy1 = SoftmaxPolicy()
    rewards1 = reinforce(env, policy1, n_episodes, alpha=2e-4)
    results_no_baseline.append(rewards1)
    
    policy2 = SoftmaxPolicy()
    rewards2 = reinforce_baseline(env, policy2, n_episodes, alpha_policy=2e-4)
    results_baseline.append(rewards2)

avg_no_baseline = np.mean(results_no_baseline, axis=0)
avg_baseline = np.mean(results_baseline, axis=0)

In [ ]:
# Plot comparison
plt.figure(figsize=(10, 5))

window = 50
smooth_no_base = np.convolve(avg_no_baseline, np.ones(window)/window, mode='valid')
smooth_base = np.convolve(avg_baseline, np.ones(window)/window, mode='valid')

plt.plot(smooth_no_base, label='REINFORCE', linewidth=2)
plt.plot(smooth_base, label='REINFORCE + Baseline', linewidth=2)
plt.axhline(y=-11.6, color='gray', linestyle='--', label='Optimal')
plt.xlabel('Episode')
plt.ylabel('Average Reward')
plt.title('REINFORCE: Effect of Baseline')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 9.6 Actor-Critic Methods

REINFORCE, **Monte Carlo** return kullanır → yüksek variance, episode bitmeli.

**Actor-Critic**: TD learning ile birleştir:
- **Actor**: Policy $\pi_\theta$ (güncellenir)
- **Critic**: Value function $V_w$ veya $Q_w$ (policy'yi değerlendirir)

### One-Step Actor-Critic

TD error'u advantage olarak kullan:

$$\delta_t = R_{t+1} + \gamma V_w(S_{t+1}) - V_w(S_t)$$

$$\theta \leftarrow \theta + \alpha_\theta \delta_t \nabla_\theta \log \pi_\theta(A_t|S_t)$$
$$w \leftarrow w + \alpha_w \delta_t \nabla_w V_w(S_t)$$

In [ ]:
def actor_critic(env, policy, n_episodes=1000, alpha_policy=2e-4, 
                 alpha_value=0.1, gamma=1.0):
    """
    One-Step Actor-Critic.
    """
    V = ValueFunction(env.n_states)
    episode_rewards = []
    
    for episode in range(n_episodes):
        state = env.reset()
        I = 1.0  # Discount for policy gradient
        total_reward = 0
        
        while True:
            action = policy.select_action(state)
            next_state, reward, done = env.step(action)
            total_reward += reward
            
            # TD error (advantage estimate)
            if done:
                delta = reward - V(state)
            else:
                delta = reward + gamma * V(next_state) - V(state)
            
            # Update critic (value function)
            V.update(state, V(state) + delta, alpha=alpha_value)
            
            # Update actor (policy)
            grad = policy.get_grad_log_prob(state, action)
            policy.theta += alpha_policy * I * delta * grad
            
            I *= gamma
            
            if done:
                break
            
            state = next_state
        
        episode_rewards.append(total_reward)
    
    return episode_rewards

# Train Actor-Critic
policy_ac = SoftmaxPolicy()
rewards_ac = actor_critic(env, policy_ac, n_episodes=1000, alpha_policy=2e-4)

print(f"Actor-Critic: Final θ = {policy_ac.theta[0]:.4f}")
print(f"P(right) = {policy_ac.get_probs()[1]:.4f}")

In [ ]:
# Compare all methods
n_runs = 20
n_episodes = 1000

all_results = {
    'REINFORCE': [],
    'REINFORCE + Baseline': [],
    'Actor-Critic': []
}

for _ in range(n_runs):
    p1 = SoftmaxPolicy()
    all_results['REINFORCE'].append(reinforce(env, p1, n_episodes, alpha=2e-4))
    
    p2 = SoftmaxPolicy()
    all_results['REINFORCE + Baseline'].append(
        reinforce_baseline(env, p2, n_episodes, alpha_policy=2e-4))
    
    p3 = SoftmaxPolicy()
    all_results['Actor-Critic'].append(
        actor_critic(env, p3, n_episodes, alpha_policy=2e-4))

# Plot
plt.figure(figsize=(12, 5))
window = 50
colors = ['blue', 'green', 'red']

for (name, results), color in zip(all_results.items(), colors):
    avg = np.mean(results, axis=0)
    smoothed = np.convolve(avg, np.ones(window)/window, mode='valid')
    plt.plot(smoothed, label=name, color=color, linewidth=2)

plt.axhline(y=-11.6, color='gray', linestyle='--', label='Optimal')
plt.xlabel('Episode')
plt.ylabel('Average Reward')
plt.title('Policy Gradient Methods Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 9.7 CartPole Example

Daha karmaşık bir environment'ta policy gradient uygulayalım.

In [ ]:
class SimpleCartPole:
    """
    Simplified CartPole environment.
    State: [position, velocity, angle, angular_velocity]
    Actions: 0 (left), 1 (right)
    """
    
    def __init__(self):
        self.gravity = 9.8
        self.masscart = 1.0
        self.masspole = 0.1
        self.length = 0.5
        self.force_mag = 10.0
        self.tau = 0.02  # Time step
        
        self.theta_threshold = 12 * np.pi / 180  # Angle limit
        self.x_threshold = 2.4  # Position limit
        
        self.n_actions = 2
        self.state_dim = 4
        self.reset()
    
    def reset(self):
        self.state = np.random.uniform(-0.05, 0.05, 4)
        return self.state.copy()
    
    def step(self, action):
        x, x_dot, theta, theta_dot = self.state
        force = self.force_mag if action == 1 else -self.force_mag
        
        costheta = np.cos(theta)
        sintheta = np.sin(theta)
        
        total_mass = self.masscart + self.masspole
        polemass_length = self.masspole * self.length
        
        temp = (force + polemass_length * theta_dot**2 * sintheta) / total_mass
        theta_acc = (self.gravity * sintheta - costheta * temp) / \
                    (self.length * (4/3 - self.masspole * costheta**2 / total_mass))
        x_acc = temp - polemass_length * theta_acc * costheta / total_mass
        
        # Euler integration
        x = x + self.tau * x_dot
        x_dot = x_dot + self.tau * x_acc
        theta = theta + self.tau * theta_dot
        theta_dot = theta_dot + self.tau * theta_acc
        
        self.state = np.array([x, x_dot, theta, theta_dot])
        
        done = (abs(x) > self.x_threshold or abs(theta) > self.theta_threshold)
        reward = 1.0 if not done else 0.0
        
        return self.state.copy(), reward, done

class LinearPolicy:
    """
    Linear softmax policy for continuous state space.
    """
    
    def __init__(self, state_dim, n_actions):
        self.state_dim = state_dim
        self.n_actions = n_actions
        # Weight matrix: (state_dim + 1) x n_actions (with bias)
        self.theta = np.zeros((state_dim + 1, n_actions))
    
    def get_features(self, state):
        """State features with bias."""
        return np.append(state, 1.0)
    
    def get_probs(self, state):
        features = self.get_features(state)
        logits = features @ self.theta
        return softmax(logits)
    
    def select_action(self, state):
        probs = self.get_probs(state)
        return np.random.choice(self.n_actions, p=probs)
    
    def get_grad_log_prob(self, state, action):
        """Gradient of log π(a|s) w.r.t. θ."""
        features = self.get_features(state)
        probs = self.get_probs(state)
        
        # Gradient for softmax: φ(s) * (I(a) - π(a))
        grad = np.zeros_like(self.theta)
        for a in range(self.n_actions):
            indicator = 1.0 if a == action else 0.0
            grad[:, a] = features * (indicator - probs[a])
        
        return grad

# Test
cartpole = SimpleCartPole()
print(f"CartPole: state_dim={cartpole.state_dim}, n_actions={cartpole.n_actions}")

In [ ]:
def reinforce_cartpole(env, n_episodes=500, alpha=0.01, gamma=0.99):
    """
    REINFORCE for CartPole.
    """
    policy = LinearPolicy(env.state_dim, env.n_actions)
    episode_lengths = []
    
    for episode in range(n_episodes):
        states, actions, rewards = [], [], []
        state = env.reset()
        
        for t in range(500):  # Max steps
            action = policy.select_action(state)
            states.append(state)
            actions.append(action)
            
            next_state, reward, done = env.step(action)
            rewards.append(reward)
            
            if done:
                break
            state = next_state
        
        episode_lengths.append(len(rewards))
        
        # Calculate returns with baseline (mean)
        T = len(rewards)
        returns = np.zeros(T)
        G = 0
        for t in range(T - 1, -1, -1):
            G = rewards[t] + gamma * G
            returns[t] = G
        
        # Normalize returns
        if len(returns) > 1:
            returns = (returns - np.mean(returns)) / (np.std(returns) + 1e-8)
        
        # Update policy
        for t in range(T):
            grad = policy.get_grad_log_prob(states[t], actions[t])
            policy.theta += alpha * (gamma ** t) * returns[t] * grad
    
    return policy, episode_lengths

# Train
print("Training REINFORCE on CartPole...")
policy_cartpole, lengths = reinforce_cartpole(cartpole, n_episodes=500, alpha=0.01)
print(f"Final average episode length: {np.mean(lengths[-50:]):.1f}")

In [ ]:
# Plot learning curve
plt.figure(figsize=(10, 5))

window = 20
smoothed = np.convolve(lengths, np.ones(window)/window, mode='valid')

plt.plot(lengths, alpha=0.3, color='blue')
plt.plot(smoothed, color='blue', linewidth=2, label='Smoothed')
plt.axhline(y=500, color='r', linestyle='--', label='Max (solved)')
plt.xlabel('Episode')
plt.ylabel('Episode Length')
plt.title('REINFORCE on CartPole')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Özet

### Policy Gradient Methods

| Yöntem | Özellik | Avantaj | Dezavantaj |
|--------|---------|---------|------------|
| **REINFORCE** | MC return | Basit | Yüksek variance |
| **+ Baseline** | V(s) çıkar | Düşük variance | Baseline öğrenmeli |
| **Actor-Critic** | TD error | Online, düşük var. | Bias riski |

### Önemli Formüller

**Policy Gradient Theorem**:
$$\nabla J(\theta) = E_{\pi}[Q^\pi(s, a) \nabla_\theta \log \pi_\theta(a|s)]$$

**REINFORCE Update**:
$$\theta \leftarrow \theta + \alpha G_t \nabla_\theta \log \pi_\theta(A_t|S_t)$$

**Actor-Critic Update**:
$$\theta \leftarrow \theta + \alpha \delta_t \nabla_\theta \log \pi_\theta(A_t|S_t)$$

### İleri Konular

- **A2C/A3C**: Asenkron actor-critic
- **PPO**: Proximal Policy Optimization
- **TRPO**: Trust Region Policy Optimization
- **SAC**: Soft Actor-Critic

Bu yöntemler, deep RL'nin temelini oluşturur!

---

# RL Tutorial Tamamlandı!

Bu 9 notebook'ta, Sutton & Barto'nun kitabını temel alarak RL'nin temellerini öğrendik:

1. **Introduction**: RL nedir, temel kavramlar
2. **Multi-Armed Bandits**: Exploration vs exploitation
3. **MDPs**: State, action, reward, Bellman equations
4. **Dynamic Programming**: Policy/value iteration
5. **Monte Carlo**: Model-free, episode-based
6. **TD Learning**: SARSA, Q-learning
7. **N-Step**: TD ve MC arası
8. **Planning**: Dyna-Q, model-based
9. **Policy Gradient**: Direct policy optimization

## Sonraki Adımlar

- Deep RL: Neural networks + RL (DQN, A3C, PPO)
- Continuous control: DDPG, TD3, SAC
- Model-based deep RL: World models, MuZero
- Multi-agent RL

İyi öğrenmeler!